# Import libraries

In [1]:
import pandas as pd
import os
import seaborn as sns
from pathlib import Path
import parquet
import fastparquet
import pyarrow

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

# Import data

In [2]:
path = r"../data/raw/Global Superstore.xlsx"


df_orders = pd.read_excel(io = path, sheet_name = "Orders")
print("Sheet name: Orders")
df_orders.info()
print("\n------------------------\n")

df_returns = pd.read_excel(io = path, sheet_name = "Returns")
print("Sheet name: Returns")
df_returns.info()
print("\n------------------------\n")

df_employees = pd.read_excel(io = path, sheet_name = "People")
print("Sheet name: People")
df_employees.info()
print("\n------------------------\n")


print(f"\n\nData successfully imported from: '{Path(path).resolve()}'")

Sheet name: Orders
<class 'pandas.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 24 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Row ID          51290 non-null  int64         
 1   Order ID        51290 non-null  str           
 2   Order Date      51290 non-null  datetime64[us]
 3   Ship Date       51290 non-null  datetime64[us]
 4   Ship Mode       51290 non-null  str           
 5   Customer ID     51290 non-null  str           
 6   Customer Name   51290 non-null  str           
 7   Segment         51290 non-null  str           
 8   City            51290 non-null  str           
 9   State           51290 non-null  str           
 10  Country         51290 non-null  str           
 11  Postal Code     9994 non-null   float64       
 12  Market          51290 non-null  str           
 13  Region          51290 non-null  str           
 14  Product ID      51290 non-null  str           

# Drop duplicates

In [3]:
def clean_duplicates(data_frame):
    duplicated_rows_count = data_frame.duplicated().sum()
    has_duplicates = data_frame.duplicated().any()

    print(f"Initial row count: {data_frame.shape[0]}")
    print(f"Duplicate rows detected: {has_duplicates}")
    print(f"Duplicate rows count: {duplicated_rows_count}")

    if has_duplicates:
        data_frame = data_frame.drop_duplicates()
        print(f"{duplicated_rows_count} duplicate rows were found.")
        print(f"Duplicate rows removed. New row count: {data_frame.shape[0]}")
    else:
        print("No duplicate rows were found.")

    return data_frame



print("Sheet name: Orders")
df_orders = clean_duplicates(df_orders)
print("\n------------------------\n")

print("Sheet name: Returns")
df_returns = clean_duplicates(df_returns)
print("\n------------------------\n")

print("Sheet name: People")
df_employees = clean_duplicates(df_employees)

Sheet name: Orders
Initial row count: 51290
Duplicate rows detected: False
Duplicate rows count: 0
No duplicate rows were found.

------------------------

Sheet name: Returns
Initial row count: 1173
Duplicate rows detected: False
Duplicate rows count: 0
No duplicate rows were found.

------------------------

Sheet name: People
Initial row count: 13
Duplicate rows detected: False
Duplicate rows count: 0
No duplicate rows were found.


# Create Unique Order ID

In [4]:
unique_order_id = df_orders["Order ID"].nunique()
total_real_orders_count = df_orders.groupby(by="Order ID")["Customer ID"].nunique().reset_index()["Customer ID"].sum()

print(f"Unique Order IDs (raw): {unique_order_id}")
print(f"Total real orders count: {total_real_orders_count}")
print(f"Data issue found: {total_real_orders_count - unique_order_id} Order IDs are recycled for different customers!")


df_orders["Unique Order ID"] = (
    df_orders["Order ID"].astype(str) + "_" + 
    df_orders["Customer ID"].astype(str) + "_" + 
    df_orders["Order Date"].astype(str)
)


print(f"Fixed Unique Order IDs: {df_orders['Unique Order ID'].nunique()}")
print("Success: Unique Order ID column created. Order collisions are fixed!")

Unique Order IDs (raw): 25035
Total real orders count: 25753
Data issue found: 718 Order IDs are recycled for different customers!
Fixed Unique Order IDs: 25754
Success: Unique Order ID column created. Order collisions are fixed!


# Fixing columns

### Columns: Postal Code (orders data frame)

In [5]:
df_united_states = df_orders.loc[df_orders["Country"] == "United States"]
null_postal_total = df_orders["Postal Code"].isnull().sum()
null_postal_us = df_united_states["Postal Code"].isnull().sum()

print(f"Total number of null observations in the Postal Code column across all countries: {null_postal_total}.")
print(f"Total number of null observations in the Postal Code column for the United States: {null_postal_us}.")
print("We can observe that the postal code is missing worldwide, except in the USA.")

df_orders["Postal Code"] = df_orders["Postal Code"].fillna("Unknown")
print(f"Successfully filled {null_postal_total} missing values in the 'Postal Code' column with 'Unknown'.")

Total number of null observations in the Postal Code column across all countries: 41296.
Total number of null observations in the Postal Code column for the United States: 0.
We can observe that the postal code is missing worldwide, except in the USA.
Successfully filled 41296 missing values in the 'Postal Code' column with 'Unknown'.


### Columns: Region (employees data frame)

In [6]:
# 'AMEA' region detected in the Employees dataset, conflicting with 'EMEA' in the Orders dataset.
# Filtering confirmed that staff assigned to European countries were incorrectly tagged as 'AMEA'.
# Identified 'AMEA' as a typographical error for 'EMEA' within the personnel records.
# Standardized 'AMEA' to 'EMEA' to ensure consistent mapping between employees and sales data.

print("List of all unique countries belonging to the EMEA region within the orders dataset. This confirms that the 'AMEA' region used in the employees dataset is a labeling error.\n")
print(f"{df_orders.loc[df_orders['Region'] == 'EMEA']['Country'].unique()}\n")


print("Employees dataframe")
df_employees["Region"] = df_employees["Region"].replace({"AMEA": "EMEA"})
print(df_employees.head(20))

List of all unique countries belonging to the EMEA region within the orders dataset. This confirms that the 'AMEA' region used in the employees dataset is a labeling error.

<ArrowStringArray>
[          'Saudi Arabia',                 'Poland',                   'Iran',
                'Ukraine',                'Belarus',                 'Russia',
             'Azerbaijan',              'Lithuania',                'Romania',
                 'Turkey',                'Hungary',                   'Iraq',
                'Georgia',                'Albania',             'Montenegro',
                'Austria',                  'Qatar',                'Estonia',
         'Czech Republic',                  'Syria',                'Lebanon',
               'Bulgaria',                 'Israel',               'Slovenia',
               'Mongolia', 'Bosnia and Herzegovina',                'Croatia',
             'Kyrgyzstan',             'Uzbekistan',                'Bahrain',
                 

### Columns: Market, Sub-Category, Product ID (orders data frame)

In [7]:
duplicate_countries = df_orders[["Country", "Market"]].drop_duplicates()["Country"].value_counts().reset_index(name="markets_count")
print(duplicate_countries[duplicate_countries["markets_count"] > 1])

print("\nFixed country-to-market mapping:")
print(" - Reassigned Austria to EU")
print(" - Reassigned Mongolia to APAC\n")

df_orders.loc[df_orders["Country"] == "Austria", "Market"] = "EU"
df_orders.loc[df_orders["Country"] == "Mongolia", "Market"] = "APAC"

    Country  markets_count
0   Austria              2
1  Mongolia              2

Fixed country-to-market mapping:
 - Reassigned Austria to EU
 - Reassigned Mongolia to APAC



In [8]:
duplicate_subcats = df_orders[["Sub-Category", "Category"]].drop_duplicates()["Sub-Category"].value_counts().reset_index(name="categories_count")
print(duplicate_subcats[duplicate_subcats["categories_count"] > 1])

print("\nNo issues found. Each sub-category belongs to a single category.")

Empty DataFrame
Columns: [Sub-Category, categories_count]
Index: []

No issues found. Each sub-category belongs to a single category.


In [9]:
duplicate_products = df_orders[["Product ID", "Sub-Category"]].drop_duplicates()["Product ID"].value_counts().reset_index(name="subcategories_count")
print(duplicate_products[duplicate_products["subcategories_count"] > 1])

print("\nFixed Product ID mapping:")
print(" - Reassigned 'Avery File Folder Labels, Adjustable' to OFF-AVE-10002103\n")

df_orders.loc[df_orders["Product Name"] == "Avery File Folder Labels, Adjustable", "Product ID"] = "OFF-AVE-10002103"

         Product ID  subcategories_count
0  OFF-AVE-10002102                    2

Fixed Product ID mapping:
 - Reassigned 'Avery File Folder Labels, Adjustable' to OFF-AVE-10002103



# Change datatypes of columns

In [10]:
df_orders["Row ID"] = df_orders["Row ID"].astype("int64")
df_orders["Order ID"] = df_orders["Order ID"].astype("str")
df_orders["Order Priority"] = df_orders["Order Priority"].astype("category")
df_orders["Order Date"] = pd.to_datetime(df_orders["Order Date"])




df_orders["Ship Date"] = pd.to_datetime(df_orders["Ship Date"])
df_orders["Ship Mode"] = df_orders["Ship Mode"].astype("category")
df_orders["Customer ID"] = df_orders["Customer ID"].astype("str")
df_orders["Customer Name"] = df_orders["Customer Name"].astype("str")
df_orders["Segment"] = df_orders["Segment"].astype("category")
df_orders["City"] = df_orders["City"].astype("str")
df_orders["State"] = df_orders["State"].astype("str")
df_orders["Country"] = df_orders["Country"].astype("str")
df_orders["Postal Code"] = df_orders["Postal Code"].astype("str")
df_orders["Market"] = df_orders["Market"].astype("category")
df_orders["Region"] = df_orders["Region"].astype("category")
df_orders["Product ID"] = df_orders["Product ID"].astype("str")
df_orders["Category"] = df_orders["Category"].astype("category")
df_orders["Sub-Category"] = df_orders["Sub-Category"].astype("category")
df_orders["Product Name"] = df_orders["Product Name"].astype("str")
df_orders["Quantity"] = df_orders["Quantity"].astype("int16")
df_orders["Discount"] = df_orders["Discount"].astype("float32")
df_orders["Sales"] = df_orders["Sales"].astype("float32")
df_orders["Profit"] = df_orders["Profit"].astype("float32")
df_orders["Shipping Cost"] = df_orders["Shipping Cost"].astype("float32")

df_orders.info()

print("\n-----------------------------------------------")
print("I optimized the dataset by converting categorical columns to the 'category' dtype and reducing numeric columns to smaller integer types where possible.\n" \
"This reduced memory usage and improved performance for groupby operations.\n")

<class 'pandas.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 25 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Row ID           51290 non-null  int64         
 1   Order ID         51290 non-null  str           
 2   Order Date       51290 non-null  datetime64[us]
 3   Ship Date        51290 non-null  datetime64[us]
 4   Ship Mode        51290 non-null  category      
 5   Customer ID      51290 non-null  str           
 6   Customer Name    51290 non-null  str           
 7   Segment          51290 non-null  category      
 8   City             51290 non-null  str           
 9   State            51290 non-null  str           
 10  Country          51290 non-null  str           
 11  Postal Code      51290 non-null  str           
 12  Market           51290 non-null  category      
 13  Region           51290 non-null  category      
 14  Product ID       51290 non-null  str           
 

# Feature Engineering & Target Engineering

In [11]:
df_orders["Delivery Time"] = (df_orders["Ship Date"] - df_orders["Order Date"]).dt.days
df_orders["Delivery Time"] = df_orders["Delivery Time"].astype("int16")

df_orders["Original Price"] = df_orders["Sales"] / (1 - df_orders["Discount"])
df_orders["Original Price"] = df_orders["Original Price"].astype("float32")

df_orders["Discount Value"] = df_orders["Original Price"] - df_orders["Sales"]
df_orders["Discount Value"] = df_orders["Discount Value"].astype("float32")

df_orders["Profit Margin"] = round(df_orders["Profit"] / df_orders["Sales"], 2)
df_orders["Profit Margin"] = df_orders["Profit Margin"].astype("float32")

df_orders["Cost"] = df_orders["Sales"] - df_orders["Profit"]
df_orders["Cost"] = df_orders["Cost"].astype("float32")

df_orders["Product Cost"] = df_orders["Cost"] - df_orders["Shipping Cost"]
df_orders["Product Cost"] = df_orders["Product Cost"].astype("float32")

df_orders["Shipping Ratio"] = df_orders["Shipping Cost"] / df_orders["Sales"]
df_orders["Shipping Ratio"] = df_orders["Shipping Ratio"].astype("float32")

df_orders["Cost Ratio"] = df_orders["Cost"] / df_orders["Sales"] 
df_orders["Cost Ratio"] = df_orders["Cost Ratio"].astype("float32")

df_orders["Order Day"] = df_orders["Order Date"].dt.day
df_orders["Order Day"] = df_orders["Order Day"].astype("int8")

df_orders["Order Day Name"] = df_orders["Order Date"].dt.day_name()
df_orders["Order Day Name"] = pd.Categorical(
                                            df_orders["Order Day Name"], 
                                            categories = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"],
                                            ordered = True
                                        )

df_orders["Is Weekend"] = False
df_orders.loc[(df_orders["Order Day Name"] == "Saturday") | (df_orders["Order Day Name"] == "Sunday"), "Is Weekend"] = True
df_orders["Is Weekend"] = df_orders["Is Weekend"].astype("boolean")

df_orders["Order Month"] = df_orders["Order Date"].dt.month_name()
df_orders["Order Month"] = pd.Categorical(
                                            df_orders["Order Month"], 
                                            categories = ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"],
                                            ordered = True
                                        )   
 
df_orders["Order Year"] = df_orders["Order Date"].dt.year
df_orders["Order Year"] = df_orders["Order Year"].astype("int16")






df_orders = df_orders[["Row ID", "Unique Order ID", "Order ID", "Order Priority", "Order Date", "Order Day", "Order Month", "Order Year", "Order Day Name", "Is Weekend", "Ship Date", 
                         "Ship Mode", "Delivery Time", "Customer ID", "Customer Name", "Segment", "City", "State",
                         "Country", "Postal Code", "Market", "Region", "Product ID", "Category", "Sub-Category", "Product Name",
                         "Quantity", "Original Price", "Discount", "Discount Value", "Sales", "Profit", "Profit Margin", "Cost", "Shipping Cost", "Product Cost", "Shipping Ratio", "Cost Ratio"]]


print("New columns have been introduced into the dataframe: Delivery Time, Original Price, Discount Value, Profit Margin, Cost, Product Cost, Shipping Ratio, Cost Ratio, Order Day, Order Day Name, Is Weekend, Order Month, Order Year")
df_orders.info()

New columns have been introduced into the dataframe: Delivery Time, Original Price, Discount Value, Profit Margin, Cost, Product Cost, Shipping Ratio, Cost Ratio, Order Day, Order Day Name, Is Weekend, Order Month, Order Year
<class 'pandas.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 38 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Row ID           51290 non-null  int64         
 1   Unique Order ID  51290 non-null  str           
 2   Order ID         51290 non-null  str           
 3   Order Priority   51290 non-null  category      
 4   Order Date       51290 non-null  datetime64[us]
 5   Order Day        51290 non-null  int8          
 6   Order Month      51290 non-null  category      
 7   Order Year       51290 non-null  int16         
 8   Order Day Name   51290 non-null  category      
 9   Is Weekend       51290 non-null  boolean       
 10  Ship Date        51290 non-null  da

# Data merging

In [12]:
df_returns["Returned"] = True
df_employees["Region"] = df_employees["Region"].astype("category")
df_employees["Person"] = df_employees["Person"].astype("str")

data_frame = pd.merge(left = df_orders, right = df_employees, how = "inner", on = "Region")
data_frame = pd.merge(left = data_frame, right = df_returns, how = "left", left_on = ["Order ID", "Market"], right_on = ["Order ID", "Market"])



print(f"Before merging with the returns dataframe: {df_orders.shape[0]} rows count")
print(f"After merging with the returns dataframe: {data_frame.shape[0]} rows count")
print(f"{data_frame.duplicated().sum()} duplicate rows were found\n")

data_frame = data_frame.drop_duplicates()

print("Due to inconsistencies in the dataset, some Order IDs were reused across different transactions. This caused unintended duplicates during the merge process.\n"
"After validating that these duplicates were identical and did not contain conflicting information, they were safely removed using drop_duplicates().")


data_frame["Returned"] = data_frame["Returned"].fillna(False)
data_frame["Returned"] = data_frame["Returned"].astype("boolean")
data_frame = data_frame.rename(columns={"Person": "Employee"})


data_frame.columns = (data_frame.columns.str.lower().str.replace(" ", "_").str.replace("-", "_"))
data_frame = data_frame.reset_index(drop=True)

Before merging with the returns dataframe: 51290 rows count
After merging with the returns dataframe: 51290 rows count
0 duplicate rows were found

Due to inconsistencies in the dataset, some Order IDs were reused across different transactions. This caused unintended duplicates during the merge process.
After validating that these duplicates were identical and did not contain conflicting information, they were safely removed using drop_duplicates().


# Exporting dataframe to excel file

In [13]:
if os.path.isdir(r"../data/processed") is False:
    os.mkdir(r"../data/processed")

with pd.ExcelWriter(r"../data/processed/Global Superstore_processed.xlsx") as writer:
    data_frame.to_excel(writer, sheet_name = "Orders", header = True, index = False)

data_frame.to_parquet(path = r"../data/processed/global_superstore_dataframe.parquet", engine = "pyarrow")

print("-----------------------------------------------\n\n")
print("The data has been processed!\n\n")
data_frame.info()
data_frame.head(5)

-----------------------------------------------


The data has been processed!


<class 'pandas.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 40 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   row_id           51290 non-null  int64         
 1   unique_order_id  51290 non-null  str           
 2   order_id         51290 non-null  str           
 3   order_priority   51290 non-null  category      
 4   order_date       51290 non-null  datetime64[us]
 5   order_day        51290 non-null  int8          
 6   order_month      51290 non-null  category      
 7   order_year       51290 non-null  int16         
 8   order_day_name   51290 non-null  category      
 9   is_weekend       51290 non-null  boolean       
 10  ship_date        51290 non-null  datetime64[us]
 11  ship_mode        51290 non-null  category      
 12  delivery_time    51290 non-null  int16         
 13  customer_id      5129

,row_id,unique_order_id,order_id,order_priority,order_date,order_day,order_month,order_year,order_day_name,is_weekend,ship_date,ship_mode,delivery_time,customer_id,customer_name,segment,city,state,country,postal_code,market,region,product_id,category,sub_category,product_name,quantity,original_price,discount,discount_value,sales,profit,profit_margin,cost,shipping_cost,product_cost,shipping_ratio,cost_ratio,employee,returned
0,32298,CA-2012-124891_RH-19495_2012-07-31,CA-2012-124891,Critical,2012-07-31,31,July,2012,Tuesday,False,2012-07-31,Same Day,0,RH-19495,Rick Hansen,Consumer,New York City,New York,United States,10024.0,US,East,TEC-AC-10003033,Technology,Accessories,Plantronics CS510 - Over-the-Head monaural Wir...,7,2309.649902,0.0,0.000000,2309.649902,762.184509,0.33,1547.465332,933.570007,613.895325,0.404204,0.670000,Kelly Williams,False
1,26341,IN-2013-77878_JR-16210_2013-02-05,IN-2013-77878,Critical,2013-02-05,5,February,2013,Tuesday,False,2013-02-07,Second Class,2,JR-16210,Justin Ritter,Corporate,Wollongong,New South Wales,Australia,Unknown,APAC,Oceania,FUR-CH-10003950,Furniture,Chairs,"Novimex Executive Leather Armchair, Black",9,4121.550293,0.1,412.155273,3709.395020,-288.765015,-0.08,3998.160156,923.630005,3074.530273,0.248997,1.077847,Anthony Jacobs,True
2,25330,IN-2013-71249_CR-12730_2013-10-17,IN-2013-71249,Medium,2013-10-17,17,October,2013,Thursday,False,2013-10-18,First Class,1,CR-12730,Craig Reiter,Consumer,Brisbane,Queensland,Australia,Unknown,APAC,Oceania,TEC-PH-10004664,Technology,Phones,"Nokia Smart Phone, with Caller ID",9,5750.189941,0.1,575.019043,5175.170898,919.971008,0.18,4255.199707,915.489990,3339.709717,0.176900,0.822234,Anthony Jacobs,False
3,13524,ES-2013-1579342_KM-16375_2013-01-28,ES-2013-1579342,Medium,2013-01-28,28,January,2013,Monday,False,2013-01-30,First Class,2,KM-16375,Katherine Murray,Home Office,Berlin,Berlin,Germany,Unknown,EU,Central,TEC-PH-10004583,Technology,Phones,"Motorola Smart Phone, Cordless",5,3213.900146,0.1,321.390137,2892.510010,-96.540001,-0.03,2989.050049,910.159973,2078.890137,0.314661,1.033376,Anna Andreadi,False
4,47221,SG-2013-4320_RH-9495_2013-11-05,SG-2013-4320,Critical,2013-11-05,5,November,2013,Tuesday,False,2013-11-06,Same Day,1,RH-9495,Rick Hansen,Consumer,Dakar,Dakar,Senegal,Unknown,Africa,Africa,TEC-SHA-10000501,Technology,Copiers,"Sharp Wireless Fax, High-Speed",8,2832.959961,0.0,0.000000,2832.959961,311.519989,0.11,2521.439941,903.039978,1618.399902,0.318762,0.890037,Deborah Brumfield,False
